In [18]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://punjabandsindbank.co.in/content/interestdom" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find_all("table")
    fd_table=table[1]
    rows=fd_table.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[2:]:  #Skip heading
        data = i.find_all("td")
        row=['Punjab and Sind Bank']
        if row_count not in {18,25}:
            row.extend([tr.text for tr in data])
            row[1] = row[1].replace('( Callable)', '').strip()
            last_element = row[-1]
            # Remove special characters ($), *, (#)
            #cleaned_element = ''.join(c for c in last_element if c.isdigit() or c == '.') + '%'
            cleaned_element = ''.join(c for c in last_element if c.isdigit() or c == '.')
            row[-1] = cleaned_element
            if row_count < 7:
                row.extend(['',str(row[2]),''])
            else:
                row.extend(['',str(float(row[2]) + 0.5),''])
            row[2] = row[2] + '%'
            row[-2] = row[-2] + '%'
            for i in range(0,5):
                row[i] = ' '.join(str(row[i]).strip().replace('\n', '').split())
            if row_count == 28: 
                tax_saver_row = ['Punjab and Sind Bank','Tax-saver Fixed Deposit',row[2],'',row[4],'']
                list_of_interest_rates.append(tax_saver_row)
             
            list_of_interest_rates.append(row)
        row_count += 1
   
    #print(list_of_interest_rates)   
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nPunjab and Sind Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 19112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_19112024.csv
               bank_name                       tenure gen_rate  \
0   Punjab and Sind Bank                  7 - 14 Days    4.00%   
1   Punjab and Sind Bank                 15 - 30 Days    4.00%   
2   Punjab and Sind Bank                 31 - 45 Days    4.25%   
3   Punjab and Sind Bank                 46 - 90 Days    4.50%   
4   Punjab and Sind Bank                91 - 120 Days    4.50%   
5   Punjab and Sind Bank                 121-150 Days    4.75%   
6   Punjab and Sind Bank               151 - 179 Days    6.00%   
7   Punjab and Sind Bank               180 – 221 Days    5.25%   
8   Punjab and Sind Bank                     222 Days    6.30%   
9   Punjab and Sind Bank               223 – 269 Days    5.25%   
10  Punjab and Sind Bank               270 – 332 Days    5.50%   
11  Punjab and Sind Bank                     333 Days    7.20%   
12  Punjab and Sind Bank              